In [ ]:
import pickle
import numpy as np
import sys
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
sys.path.append('../../')
out_path = 'drugs_noH_1000_kabsch_traj_stack_pretrain_fs_evalout.pkl'
with open(out_path, 'rb') as f:
    data = pickle.load(f)
    print(data.keys())  # smiles
smiles = list(data.keys())
print(len(data.keys()))  # 1000

In [ ]:
data

In [ ]:
data['C[C@@H](O)[C@@H](O)[C@@H](N)C#N'].keys()

In [ ]:
data['C[C@@H](O)[C@@H](O)[C@@H](N)C#N']['ref_metastable_p']

In [ ]:
data['C[C@@H](O)[C@@H](O)[C@@H](N)C#N']['gen_metastable_p']

In [ ]:
# Plot the distributions of the metastable probabilities
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1,2, figsize=(10, 4))
for smile in smiles:
    if "ref_metastable_p" in data[smile]:
        axs[0].scatter(data[smile]['ref_metastable_p'], data[smile]['gen_metastable_p'], s=5, color='blue', alpha=0.5)
        axs[1].scatter(data[smile]['ref_metastable_p'], data[smile]['md_metastable_p'], s=5, color='orange', alpha=0.5)
axs[0].set_title('ref vs gen')
axs[1].set_title('ref vs md')
axs[0].set_xlabel('ref')
axs[0].set_ylabel('gen')
axs[1].set_xlabel('ref')
axs[1].set_ylabel('md')
axs[0].set_xlim(0, 1)
axs[0].set_ylim(0, 1)
axs[1].set_xlim(0, 1)
axs[1].set_ylim(0, 1)
plt.show()

In [ ]:
bond_angle_jsd = []
bond_angle_jsd_md = []
bond_angle_jsd_top1 = []
bond_angle_jsd_top3 = []
bond_angle_jsd_top5 = []
bond_angle_jsd_top10 = []
bond_angle_w1 = []
bond_angle_w1_md = []
bond_length_jsd = []
bond_length_jsd_top1 = []
bond_length_jsd_top3 = []
bond_length_jsd_top5 = []
bond_length_jsd_top10 = []
bond_length_jsd_md = []
bond_length_w1 = []
bond_length_w1_md = []
torsion_jsd = []
torsion_jsd_top1 = []
torsion_jsd_top3 = []
torsion_jsd_top5 = []
torsion_jsd_top10 = []
torsion_jsd_md = []
torsion_w1 = []
torsion_w1_md = []
tica_0_jsd = []
tica_0_jsd_top1 = []
tica_0_jsd_top3 = []
tica_0_jsd_top5 = []
tica_0_jsd_top10 = []
tica_0_jsd_md = []
tica_0_w1 = []
tica_0_w1_md = []
tica_01_jsd = []
tica_01_jsd_top1 = []
tica_01_jsd_top3 = []
tica_01_jsd_top5 = []
tica_01_jsd_top10 = []
tica_01_jsd_md = []
decorrelation_jsd = []
decorrelation_jsd_md = []
decorrelation_w1 = []
decorrelation_w1_md = []
msm_gen_jsd = []
msm_gen_jsd_md = []
msm_gen_w1 = []
msm_gen_w1_md = []

In [ ]:
def safe_get(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d

def top_n_keys_from_sorted_list(sorted_list, n):
    return [entry[0] for entry in sorted_list[:n]] if len(sorted_list) >= n else []

no_tica = 0
no_angle = 0

for smile in smiles:
    smile_data = data.get(smile, {})
    
    if 'JSD_bond_angle_gen' not in smile_data:
        no_angle += 1
        continue

    bond_angle_jsd += smile_data['JSD_bond_angle_gen']
    bond_angle_jsd_md += smile_data['JSD_bond_angle_md']
    bond_angle_w1 += smile_data['W1_bond_angle_gen']
    bond_angle_w1_md += smile_data['W1_bond_angle_md']
    bond_length_jsd += smile_data['JSD_bond_length_gen']
    bond_length_jsd_md += smile_data['JSD_bond_length_md']
    bond_length_w1 += smile_data['W1_bond_length_gen']
    bond_length_w1_md += smile_data['W1_bond_length_md']
    torsion_jsd += list(smile_data['JSD_torsion_gen'].values())
    torsion_jsd_md += list(smile_data['JSD_torsion_md'].values())
    torsion_w1 += list(smile_data['W1_torsion_gen'].values())
    torsion_w1_md += list(smile_data['W1_torsion_md'].values())

    # Decorrelations
    decorrelation_gen = list(smile_data.get('decorrelation_gen_in_ps', {}).values())
    decorrelation_ref = list(smile_data.get('decorrelation_ref_in_ps', {}).values())
    decorrelation_md = list(smile_data.get('decorrelation_md_in_ps', {}).values())

    for dg, dr, dm in zip(decorrelation_gen, decorrelation_ref, decorrelation_md):
        gen_p, _ = np.histogram(np.array(dg), range=(5, 1380), bins=275)
        ref_p, _ = np.histogram(np.array(dr), range=(5, 1380), bins=275)
        md_p, _  = np.histogram(np.array(dm), range=(5, 1380), bins=275)

        if np.sum(ref_p) == 0:
            continue

        decorrelation_jsd.append(jensenshannon(ref_p / np.sum(ref_p), gen_p / np.sum(gen_p)))
        decorrelation_jsd_md.append(jensenshannon(ref_p / np.sum(ref_p), md_p / np.sum(md_p)))
        decorrelation_w1.append(wasserstein_distance(dg, dr))
        decorrelation_w1_md.append(wasserstein_distance(dg, dm))

    # Sorted list for top-N
    jsd_torsion_dict = smile_data.get('JSD_torsion_gen_single_traj', {})
    sorted_jsd_torsion_gen_single_traj = sorted(
        [(k, np.mean(v)) for k, v in jsd_torsion_dict.items() if isinstance(v, list) and v],
        key=lambda x: x[1]
    )
    sorted_keys = [x[0] for x in sorted_jsd_torsion_gen_single_traj]

    for top_n in [1, 3, 5, 10]:
        if len(sorted_keys) < top_n:
            continue
        top_keys = sorted_keys[:top_n]
        for k in top_keys:
            torsion_jsd_top = data[smile]['JSD_torsion_gen_single_traj'].get(k, [])
            if top_n == 1: torsion_jsd_top1 += torsion_jsd_top
            if top_n == 3: torsion_jsd_top3 += torsion_jsd_top
            if top_n == 5: torsion_jsd_top5 += torsion_jsd_top
            if top_n == 10: torsion_jsd_top10 += torsion_jsd_top

            bond_len = smile_data.get('JSD_bond_length_gen_single_traj', {}).get(k, [])
            if top_n == 1: bond_length_jsd_top1 += bond_len
            if top_n == 3: bond_length_jsd_top3 += bond_len
            if top_n == 5: bond_length_jsd_top5 += bond_len
            if top_n == 10: bond_length_jsd_top10 += bond_len

            bond_ang = smile_data.get('JSD_bond_angle_gen_single_traj', {}).get(k, [])
            bond_ang_filtered = [x for x in bond_ang if x != -1]
            if top_n == 1: bond_angle_jsd_top1 += bond_ang_filtered
            if top_n == 3: bond_angle_jsd_top3 += bond_ang_filtered
            if top_n == 5: bond_angle_jsd_top5 += bond_ang_filtered
            if top_n == 10: bond_angle_jsd_top10 += bond_ang_filtered

    # TICA
    if 'JSD_TICA_gen' in smile_data:
        tica_data = smile_data['JSD_TICA_gen']
        tica_0 = tica_data.get('TICA-0')
        if tica_0 is not None:
            tica_0_jsd.append(tica_0)
            tica_0_jsd_md.append(smile_data['JSD_TICA_md']['TICA-0'])
            tica_0_w1.append(smile_data['W1_TICA_gen']['TICA-0'])
            tica_0_w1_md.append(smile_data['W1_TICA_md']['TICA-0'])

            traj_dict = smile_data['JSD_TICA_gen_single_traj'].get('TICA-0', {})
            for top_n in [1, 3, 5, 10]:
                if len(sorted_keys) < top_n:
                    continue
                for k in sorted_keys[:top_n]:
                    val = traj_dict.get(k)
                    if val is not None:
                        if top_n == 1: tica_0_jsd_top1 += val
                        if top_n == 3: tica_0_jsd_top3 += val
                        if top_n == 5: tica_0_jsd_top5 += val
                        if top_n == 10: tica_0_jsd_top10 += val

        if 'TICA-0,1' in tica_data:
            tica_01_jsd.append(tica_data['TICA-0,1'])
            tica_01_jsd_md.append(smile_data['JSD_TICA_md']['TICA-0,1'])
            traj_dict = smile_data['JSD_TICA_gen_single_traj'].get('TICA-0,1', {})
            for top_n in [1, 3, 5, 10]:
                if len(sorted_keys) < top_n:
                    continue
                for k in sorted_keys[:top_n]:
                    val = traj_dict.get(k)
                    if val is not None:
                        if top_n == 1: tica_01_jsd_top1 += val
                        if top_n == 3: tica_01_jsd_top3 += val
                        if top_n == 5: tica_01_jsd_top5 += val
                        if top_n == 10: tica_01_jsd_top10 += val
    else:
        no_tica += 1

print('num of molecules with no tica:', no_tica)
print('num of molecules with no angle:', no_angle)


In [ ]:
import csv
import numpy as np

def compute_stats(arr):
    """Safely compute stats for a list or return '-' if empty."""
    if not arr:
        return ['-'] * 5
    return [round(np.mean(arr), 5),
            round(np.std(arr), 5),
            round(np.median(arr), 5),
            round(np.min(arr), 5),
            round(np.max(arr), 5)]

def compute_stats_pair(arr1, arr2):
    """Compute stats for a no_md/md pair of lists."""
    return compute_stats(arr1) + compute_stats(arr2)

header = ['List Name', 'Mean (no md)', 'Std (no md)', 'Median (no md)', 'Min (no md)', 'Max (no md)', 
          'Mean (md)', 'Std (md)', 'Median (md)', 'Min (md)', 'Max (md)']

data_to_save = [
    ['bond_angle_jsd'] + compute_stats_pair(bond_angle_jsd, bond_angle_jsd_md),
    ['bond_angle_jsd_top1'] + compute_stats(bond_angle_jsd_top1),
    ['bond_angle_jsd_top3'] + compute_stats(bond_angle_jsd_top3),
    ['bond_angle_jsd_top5'] + compute_stats(bond_angle_jsd_top5),
    ['bond_angle_jsd_top10'] + compute_stats(bond_angle_jsd_top10),

    ['bond_length_jsd'] + compute_stats_pair(bond_length_jsd, bond_length_jsd_md),
    ['bond_length_jsd_top1'] + compute_stats(bond_length_jsd_top1),
    ['bond_length_jsd_top3'] + compute_stats(bond_length_jsd_top3),
    ['bond_length_jsd_top5'] + compute_stats(bond_length_jsd_top5),
    ['bond_length_jsd_top10'] + compute_stats(bond_length_jsd_top10),

    ['torsion_jsd'] + compute_stats_pair(torsion_jsd, torsion_jsd_md),
    ['torsion_jsd_top1'] + compute_stats(torsion_jsd_top1),
    ['torsion_jsd_top3'] + compute_stats(torsion_jsd_top3),
    ['torsion_jsd_top5'] + compute_stats(torsion_jsd_top5),
    ['torsion_jsd_top10'] + compute_stats(torsion_jsd_top10),

    ['tica_0_jsd'] + compute_stats_pair(tica_0_jsd, tica_0_jsd_md),
    ['tica_0_jsd_top1'] + compute_stats(tica_0_jsd_top1),
    ['tica_0_jsd_top3'] + compute_stats(tica_0_jsd_top3),
    ['tica_0_jsd_top5'] + compute_stats(tica_0_jsd_top5),
    ['tica_0_jsd_top10'] + compute_stats(tica_0_jsd_top10),

    ['tica_01_jsd'] + compute_stats_pair(tica_01_jsd, tica_01_jsd_md),
    ['tica_01_jsd_top1'] + compute_stats(tica_01_jsd_top1),
    ['tica_01_jsd_top3'] + compute_stats(tica_01_jsd_top3),
    ['tica_01_jsd_top5'] + compute_stats(tica_01_jsd_top5),
    ['tica_01_jsd_top10'] + compute_stats(tica_01_jsd_top10),

    ['torsion_w1'] + compute_stats_pair(torsion_w1, torsion_w1_md),
    ['bond_angle_w1'] + compute_stats_pair(bond_angle_w1, bond_angle_w1_md),
    ['bond_length_w1'] + compute_stats_pair(bond_length_w1, bond_length_w1_md),
    ['tica_0_w1'] + compute_stats_pair(tica_0_w1, tica_0_w1_md),

    # Optional metrics you can uncomment
    ['decorrelation_jsd'] + compute_stats_pair(decorrelation_jsd, decorrelation_jsd_md),
    ['decorrelation_w1'] + compute_stats_pair(decorrelation_w1, decorrelation_w1_md),
    ['msm_gen_jsd'] + compute_stats_pair(msm_gen_jsd, msm_gen_jsd_md),
    ['msm_gen_w1'] + compute_stats_pair(msm_gen_w1, msm_gen_w1_md),
]

# CSV write
csv_file = f'{out_path}_statistics.csv'
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(data_to_save)

print(f"Statistics saved to {csv_file}")


In [ ]:
# plot box-plot of bond_angle_jsd, bond_angle_jsd_md, bond_angle_w1, bond_angle_w1_md, torsion_jsd, torsion_jsd_md, torsion_w1, torsion_w1_md, tica_0_jsd, tica_0_jsd_md, tica_0_w1, tica_0_w1_md, tica_01_jsd, tica_01_jsd_md
import matplotlib.pyplot as plt
import pandas as pd
plt.figure(figsize=(18, 6))
plt.boxplot([bond_angle_jsd, bond_angle_jsd_md, bond_angle_w1, bond_angle_w1_md, torsion_jsd, torsion_jsd_md, torsion_w1, torsion_w1_md, tica_0_jsd, tica_0_jsd_md, tica_0_w1, tica_0_w1_md, tica_01_jsd, tica_01_jsd_md], labels=['bond_angle_jsd', 'bond_angle_jsd_md', 'bond_angle_w1', 'bond_angle_w1_md', 'torsion_jsd', 'torsion_jsd_md', 'torsion_w1', 'torsion_w1_md', 'tica_0_jsd', 'tica_0_jsd_md', 'tica_0_w1', 'tica_0_w1_md', 'tica_01_jsd', 'tica_01_jsd_md'])
plt.show()

In [ ]:
data['N[C@H](C(=O)O)[C@@H](N)CO']['W1_torsion_md']

In [ ]:
data['N[C@H](C(=O)O)[C@@H](N)CO']['JSD_torsion_gen']

In [ ]:
data['CCC(C#N)(CO)CO']['JSD_torsion_md']

In [ ]:
data['CCC(C#N)(CO)CO']['W1_bond_angle_gen']

In [ ]:
data['CCC(C#N)(CO)CO']['JSD_TICA_gen']

In [ ]:
np.array(list(data['C[C@@H](O)[C@@H](O)[C@@H](N)C#N']['decorrelation_ref_in_ps'].values()))

In [ ]:
data['CCC(C#N)(CO)CO']['JSD_bond_angle_gen_single_traj']

In [ ]:
import pickle
import numpy as np
import sys
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
import csv
from itertools import chain

sys.path.append('../../')

out_path = "mdgen_mdgen_noH_1000_interpolator_pretrain_fs_normal_evalout.pkl"

with open(out_path, 'rb') as f:
    data = pickle.load(f)
    print(data.keys())  # smiles

smiles = list(data.keys())
print(f"Total molecules: {len(data.keys())}")

# Initialize lists for all metrics
# TORSION (Level 2 & 3 Collection)
backbone_phi_jsd = []
backbone_phi_w1 = []
backbone_psi_jsd = []
backbone_psi_w1 = []
sidechain_chi1_jsd = []
sidechain_chi1_w1 = []
sidechain_chi2_jsd = []
sidechain_chi2_w1 = []
sidechain_chi3_jsd = []
sidechain_chi3_w1 = []
sidechain_chi4_jsd = []
sidechain_chi4_w1 = []

# New lists for Overall Torsion Aggregation (L3)
all_torsion_jsd_l3 = []
all_torsion_w1_l3 = []
all_backbone_jsd_l3 = []
all_backbone_w1_l3 = []
all_sidechain_jsd_l3 = []
all_sidechain_w1_l3 = []


# BOND ANGLES/LENGTHS (Level 1 Collection, L3 Aggregation)
bond_angle_jsd = []
bond_angle_w1 = []
bond_length_jsd = []
bond_length_w1 = []

# TICA metrics (Level 2 Collection, L3 Aggregation)
tica_0_jsd = []
tica_0_w1 = []
tica_01_jsd = []

# Decorrelation (Level 2 Collection, L3 Aggregation)
decorrelation_jsd = []
decorrelation_w1 = []

# MSM metrics (Level 2 Collection, L3 Aggregation)
msm_gen_jsd = []
msm_gen_w1 = []

def safe_get(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d

no_tica = 0
no_angle = 0
no_backbone = 0

for smile in smiles:
    smile_data = data.get(smile, {})
    
    # Skip if no data
    if not smile_data:
        continue

    # --- TORSION ANGLE COLLECTION (Level 2 & L3) ---
    current_peptide_jsd = []
    current_peptide_w1 = []
    current_peptide_bb_jsd = []
    current_peptide_bb_w1 = []
    current_peptide_sc_jsd = []
    current_peptide_sc_w1 = []
    
    # Backbone angles (NEW for tetrapeptides)
    if 'JSD_backbone_gen' in smile_data:
        backbone_jsd = smile_data['JSD_backbone_gen']
        backbone_w1 = smile_data['W1_backbone_gen']
        
        if 'phi' in backbone_jsd:
            backbone_phi_jsd.append(backbone_jsd['phi'])
            backbone_phi_w1.append(backbone_w1['phi'])
            current_peptide_bb_jsd.append(backbone_jsd['phi'])
            current_peptide_bb_w1.append(backbone_w1['phi'])
        if 'psi' in backbone_jsd:
            backbone_psi_jsd.append(backbone_jsd['psi'])
            backbone_psi_w1.append(backbone_w1['psi'])
            current_peptide_bb_jsd.append(backbone_jsd['psi'])
            current_peptide_bb_w1.append(backbone_w1['psi'])
    else:
        no_backbone += 1
    
    # Sidechain angles (NEW for tetrapeptides)
    if 'JSD_sidechain_gen' in smile_data:
        sidechain_jsd = smile_data['JSD_sidechain_gen']
        sidechain_w1 = smile_data['W1_sidechain_gen']
        
        if 'chi1' in sidechain_jsd:
            sidechain_chi1_jsd.append(sidechain_jsd['chi1'])
            sidechain_chi1_w1.append(sidechain_w1['chi1'])
            current_peptide_sc_jsd.append(sidechain_jsd['chi1'])
            current_peptide_sc_w1.append(sidechain_w1['chi1'])
        if 'chi2' in sidechain_jsd:
            sidechain_chi2_jsd.append(sidechain_jsd['chi2'])
            sidechain_chi2_w1.append(sidechain_w1['chi2'])
            current_peptide_sc_jsd.append(sidechain_jsd['chi2'])
            current_peptide_sc_w1.append(sidechain_w1['chi2'])
        if 'chi3' in sidechain_jsd:
            sidechain_chi3_jsd.append(sidechain_jsd['chi3'])
            sidechain_chi3_w1.append(sidechain_w1['chi3'])
            current_peptide_sc_jsd.append(sidechain_jsd['chi3'])
            current_peptide_sc_w1.append(sidechain_w1['chi3'])
        if 'chi4' in sidechain_jsd:
            sidechain_chi4_jsd.append(sidechain_jsd['chi4'])
            sidechain_chi4_w1.append(sidechain_w1['chi4'])
            current_peptide_sc_jsd.append(sidechain_jsd['chi4'])
            current_peptide_sc_w1.append(sidechain_w1['chi4'])

    # Aggregate all torsion metrics for this peptide
    current_peptide_jsd = current_peptide_bb_jsd + current_peptide_sc_jsd
    current_peptide_w1 = current_peptide_bb_w1 + current_peptide_sc_w1
    
    if current_peptide_jsd:
        all_torsion_jsd_l3.extend(current_peptide_jsd)
        all_torsion_w1_l3.extend(current_peptide_w1)
    if current_peptide_bb_jsd:
        all_backbone_jsd_l3.extend(current_peptide_bb_jsd)
        all_backbone_w1_l3.extend(current_peptide_bb_w1)
    if current_peptide_sc_jsd:
        all_sidechain_jsd_l3.extend(current_peptide_sc_jsd)
        all_sidechain_w1_l3.extend(current_peptide_sc_w1)
        
    # --- BOND ANGLE/LENGTH COLLECTION (Level 1) ---
    if 'JSD_bond_angle_gen' in smile_data:
        # These metrics collect the JSD/W1 for every individual bond/angle across all peptides
        bond_angle_jsd += smile_data['JSD_bond_angle_gen']
        bond_angle_w1 += smile_data['W1_bond_angle_gen']
        bond_length_jsd += smile_data['JSD_bond_length_gen']
        bond_length_w1 += smile_data['W1_bond_length_gen']
    else:
        no_angle += 1
    
    # --- DECORRELATION ---
    decorrelation_gen = list(smile_data.get('decorrelation_gen_in_ps', {}).values())
    decorrelation_ref = list(smile_data.get('decorrelation_ref_in_ps', {}).values())
    
    for dg_list, dr_value in zip(decorrelation_gen, decorrelation_ref):
        if isinstance(dg_list, list):
            dg_array = np.array([x for x in dg_list if x > 0])
            if len(dg_array) > 0 and dr_value > 0:
                decorrelation_w1.append(wasserstein_distance(dg_array, [dr_value] * len(dg_array)))
                
                # JSD calculation for decorrelation (requires consistent range and bins)
                gen_p, _ = np.histogram(dg_array, range=(5, 1380), bins=275)
                ref_p, _ = np.histogram([dr_value], range=(5, 1380), bins=275)
                if np.sum(ref_p) > 0 and np.sum(gen_p) > 0:
                    decorrelation_jsd.append(jensenshannon(ref_p / np.sum(ref_p), gen_p / np.sum(gen_p)))
    
    # --- TICA & MSM ---
    if 'JSD_TICA_gen' in smile_data:
        tica_data = smile_data['JSD_TICA_gen']
        tica_0 = tica_data.get('TICA-0')
        if tica_0 is not None:
            tica_0_jsd.append(tica_0)
            tica_0_w1.append(smile_data['W1_TICA_gen']['TICA-0'])
        
        if 'TICA-0,1' in tica_data:
            tica_01_jsd.append(tica_data['TICA-0,1'])
        
        if 'JSD_msm_gen' in smile_data:
            msm_gen_jsd.append(smile_data['JSD_msm_gen'])
            msm_gen_w1.append(smile_data['W1_msm_gen'])
    else:
        no_tica += 1

print(f'Number of molecules with no TICA: {no_tica}')
print(f'Number of molecules with no bond angle: {no_angle}')
print(f'Number of molecules with no backbone: {no_backbone}')

# ============================================
# Statistics computation and CSV export
# ============================================

def compute_stats(arr):
    """Safely compute stats for a list or return '-' if empty."""
    if not arr:
        return ['-'] * 5
    # Ensure all list elements are float/number before computing
    arr = [x for x in arr if isinstance(x, (int, float))] 
    if not arr:
        return ['-'] * 5
        
    return [round(np.mean(arr), 5),
            round(np.std(arr), 5),
            round(np.median(arr), 5),
            round(np.min(arr), 5),
            round(np.max(arr), 5)]

# Simplified header (no MD baseline)
header = ['Metric', 'Mean', 'Std', 'Median', 'Min', 'Max']

data_to_save = [
    # --- OVERALL TORSION (NEW AGGREGATION) ---
    ['Overall_Torsion_JSD'] + compute_stats(all_torsion_jsd_l3),
    ['Overall_Torsion_W1'] + compute_stats(all_torsion_w1_l3),
    ['Overall_Backbone_JSD'] + compute_stats(all_backbone_jsd_l3),
    ['Overall_Backbone_W1'] + compute_stats(all_backbone_w1_l3),
    ['Overall_Sidechain_JSD'] + compute_stats(all_sidechain_jsd_l3),
    ['Overall_Sidechain_W1'] + compute_stats(all_sidechain_w1_l3),
    
    # --- INDIVIDUAL TORSION ANGLES (Level 2 Aggregation) ---
    ['backbone_phi_jsd'] + compute_stats(backbone_phi_jsd),
    ['backbone_phi_w1'] + compute_stats(backbone_phi_w1),
    ['backbone_psi_jsd'] + compute_stats(backbone_psi_jsd),
    ['backbone_psi_w1'] + compute_stats(backbone_psi_w1),
    
    ['sidechain_chi1_jsd'] + compute_stats(sidechain_chi1_jsd),
    ['sidechain_chi1_w1'] + compute_stats(sidechain_chi1_w1),
    ['sidechain_chi2_jsd'] + compute_stats(sidechain_chi2_jsd),
    ['sidechain_chi2_w1'] + compute_stats(sidechain_chi2_w1),
    ['sidechain_chi3_jsd'] + compute_stats(sidechain_chi3_jsd),
    ['sidechain_chi3_w1'] + compute_stats(sidechain_chi3_w1),
    ['sidechain_chi4_jsd'] + compute_stats(sidechain_chi4_jsd),
    ['sidechain_chi4_w1'] + compute_stats(sidechain_chi4_w1),
    
    # --- BOND ANGLES and LENGTHS (Level 1 Aggregation) ---
    ['bond_angle_jsd'] + compute_stats(bond_angle_jsd),
    ['bond_angle_w1'] + compute_stats(bond_angle_w1),
    ['bond_length_jsd'] + compute_stats(bond_length_jsd),
    ['bond_length_w1'] + compute_stats(bond_length_w1),
    
    # --- TICA ---
    ['tica_0_jsd'] + compute_stats(tica_0_jsd),
    ['tica_0_w1'] + compute_stats(tica_0_w1),
    ['tica_01_jsd'] + compute_stats(tica_01_jsd),
    
    # --- DECORRELATION ---
    ['decorrelation_jsd'] + compute_stats(decorrelation_jsd),
    ['decorrelation_w1'] + compute_stats(decorrelation_w1),
    
    # --- MSM ---
    ['msm_gen_jsd'] + compute_stats(msm_gen_jsd),
    ['msm_gen_w1'] + compute_stats(msm_gen_w1),
]

# CSV write
csv_file = f'{out_path}_statistics.csv'
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(data_to_save)

print(f"Statistics saved to {csv_file}")